# ScamStream — Two-Stage Detection: Binary + Multiclass (Kaggle)

**Pipeline:**
1. **Stage 1 — Binary model** (scam vs harmless): hybrid detection loss, per-turn streaming
2. **Stage 2 — Multiclass model** (scenario A/B/C/D): supervised contrastive loss + CE, per-conversation
   - Chỉ chạy khi Stage 1 dự đoán là **scam**

**Data cần upload lên Kaggle** (dạng dataset):
```
scamstream-data/
  train/  ← Viscam.json, Vanilla.json, real1.json, ...
  test/   ← real2.json
```
Format mỗi file JSON: list of `{label, scenario, turns}`

In [ ]:
# Install if needed
!pip install -q transformers sentencepiece scikit-learn

In [ ]:
import os, json, random, time
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

torch.backends.cuda.matmul.allow_tf32 = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Configuration

In [ ]:
@dataclass
class Config:
    # Backbone
    model_name: str = "contextboxai/halong_embedding"
    max_turn_len: int = 144
    max_turns: int = 20

    # Model
    hidden_dim: int = 256
    attn_heads: int = 8
    dropout: float = 0.2

    # Binary loss
    w_max: float = 3.0
    w_floor: float = 0.1
    patience_weight: float = 0.5
    class_weight_harmless: float = 1.0

    # Multiclass contrastive loss
    contrastive_temp: float = 0.07   # temperature tau
    contrastive_alpha: float = 0.5   # alpha*SupCon + (1-alpha)*CE

    # Augmentation
    truncate_aug: bool = True
    aug_k: int = 3
    aug_min_turns: int = 3

    # Binary training
    bin_batch_size: int = 4
    bin_grad_accum: int = 8
    bin_lr: float = 2e-5
    bin_weight_decay: float = 1e-2
    bin_grad_clip: float = 1.0
    bin_warmup_ratio: float = 0.1
    bin_epochs: int = 15
    bin_unfreeze_epoch: int = 3
    bin_patience: int = 5

    # Multiclass training
    mc_batch_size: int = 8
    mc_grad_accum: int = 8
    mc_lr: float = 3e-4
    mc_weight_decay: float = 1e-2
    mc_grad_clip: float = 1.0
    mc_warmup_ratio: float = 0.1
    mc_epochs: int = 20
    mc_unfreeze_epoch: int = 5
    mc_patience: int = 8

    # Inference
    binary_threshold: float = 0.5
    seed: int = 42
    val_ratio: float = 0.20
    output_dir: str = "/kaggle/working"

CFG = Config()

## 2. Data Paths — chỉnh sửa tại đây

In [ ]:
# Upload JSON files lên Kaggle dataset, rồi sửa path bên dưới
TRAIN_FILES = [
    "/kaggle/input/models/an05112002/exp-an/tensorflow2/default/1/exp/exp_an/dataset_scamstream-main/exp1_prompt_vs_viscam/train/Vanilla.json",
    # "/kaggle/input/scamstream-data/train/Vanilla.json",
    # "/kaggle/input/scamstream-data/train/real1.json",
]
TEST_FILE = "/kaggle/input/models/an05112002/exp-an/tensorflow2/default/1/exp/exp_an/dataset_scamstream-main/exp1_prompt_vs_viscam/test/real2.json"

for f in TRAIN_FILES + [TEST_FILE]:
    status = "OK" if os.path.exists(f) else "NOT FOUND"
    print(f"[{status}] {f}")

## 3. Utilities

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path: str) -> List[Dict]:
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def truncate_augment(dialogues: List[Dict], k: int, min_turns: int) -> List[Dict]:
    """Generate k sub-window augmentations per dialogue."""
    augmented = []
    for dlg in dialogues:
        augmented.append(dlg)
        n = len(dlg["turns"])
        candidates = [
            (s, e)
            for s in range(n)
            for e in range(s + min_turns, n + 1)
            if not (s == 0 and e == n)
        ]
        if not candidates:
            continue
        selected = random.sample(candidates, min(k, len(candidates)))
        for start, end in selected:
            augmented.append({**dlg, "turns": dlg["turns"][start:end]})
    return augmented


def stratified_val_split(
    dialogues: List[Dict],
    val_ratio: float = 0.2,
    seed: int = 42,
    label_fn=None,
) -> Tuple[List[Dict], List[Dict]]:
    if label_fn is None:
        label_fn = lambda d: d["label"]
    groups = defaultdict(list)
    for dlg in dialogues:
        groups[label_fn(dlg)].append(dlg)
    rng = random.Random(seed)
    train_out, val_out = [], []
    for group in groups.values():
        rng.shuffle(group)
        n_val = max(1, int(len(group) * val_ratio))
        val_out.extend(group[:n_val])
        train_out.extend(group[n_val:])
    rng.shuffle(train_out)
    rng.shuffle(val_out)
    return train_out, val_out


def build_scenario_map(dialogues: List[Dict]) -> Dict[str, int]:
    """Map scam scenario letters to integer indices (sorted)."""
    scenarios = sorted({
        d["scenario"]
        for d in dialogues
        if d["label"] == "scam" and d.get("scenario") is not None
    })
    return {s: i for i, s in enumerate(scenarios)}

## 4. Datasets

In [ ]:
class BinaryDialogueDataset(Dataset):
    """Dialogue dataset with binary label (scam=1, harmless=0)."""

    def __init__(self, dialogues, tokenizer, max_turn_len, max_turns):
        self.dialogues = dialogues
        self.tok = tokenizer
        self.max_turn_len = max_turn_len
        self.max_turns = max_turns

    def __len__(self):
        return len(self.dialogues)

    def _encode_dialogue(self, dlg):
        turns = dlg["turns"][:self.max_turns]
        n_real = len(turns)
        ids_list, mask_list = [], []
        for turn in turns:
            enc = self.tok(
                turn, max_length=self.max_turn_len,
                padding="max_length", truncation=True, return_tensors="pt"
            )
            ids_list.append(enc["input_ids"].squeeze(0))
            mask_list.append(enc["attention_mask"].squeeze(0))
        pad_ids  = torch.zeros(self.max_turn_len, dtype=torch.long)
        pad_mask = torch.zeros(self.max_turn_len, dtype=torch.long)
        for _ in range(self.max_turns - n_real):
            ids_list.append(pad_ids)
            mask_list.append(pad_mask)
        turn_mask = torch.zeros(self.max_turns, dtype=torch.bool)
        turn_mask[:n_real] = True
        return torch.stack(ids_list), torch.stack(mask_list), turn_mask, n_real

    def __getitem__(self, idx):
        dlg = self.dialogues[idx]
        ids, masks, tmask, n_real = self._encode_dialogue(dlg)
        return {
            "input_ids":  ids,
            "attn_masks": masks,
            "turn_mask":  tmask,
            "n_turns":    torch.tensor(n_real),
            "label":      torch.tensor(1 if dlg["label"] == "scam" else 0, dtype=torch.long),
        }


class ScamMulticlassDataset(Dataset):
    """
    Dataset for multiclass scenario classification.
    Chỉ chứa scam dialogues. Label = scenario index.
    Prediction là per-conversation (không phải per-turn).
    """

    def __init__(self, dialogues, tokenizer, max_turn_len, max_turns, scenario_map):
        self.dialogues = [
            d for d in dialogues
            if d["label"] == "scam" and d.get("scenario") in scenario_map
        ]
        self.tok = tokenizer
        self.max_turn_len = max_turn_len
        self.max_turns = max_turns
        self.scenario_map = scenario_map

    def __len__(self):
        return len(self.dialogues)

    def __getitem__(self, idx):
        dlg = self.dialogues[idx]
        turns = dlg["turns"][:self.max_turns]
        n_real = len(turns)
        ids_list, mask_list = [], []
        for turn in turns:
            enc = self.tok(
                turn, max_length=self.max_turn_len,
                padding="max_length", truncation=True, return_tensors="pt"
            )
            ids_list.append(enc["input_ids"].squeeze(0))
            mask_list.append(enc["attention_mask"].squeeze(0))
        pad_ids  = torch.zeros(self.max_turn_len, dtype=torch.long)
        pad_mask = torch.zeros(self.max_turn_len, dtype=torch.long)
        for _ in range(self.max_turns - n_real):
            ids_list.append(pad_ids)
            mask_list.append(pad_mask)
        turn_mask = torch.zeros(self.max_turns, dtype=torch.bool)
        turn_mask[:n_real] = True
        return {
            "input_ids":  torch.stack(ids_list),
            "attn_masks": torch.stack(mask_list),
            "turn_mask":  turn_mask,
            "n_turns":    torch.tensor(n_real),
            "label":      torch.tensor(self.scenario_map[dlg["scenario"]], dtype=torch.long),
        }


def collate_fn(batch):
    return {
        "input_ids":  torch.stack([b["input_ids"]  for b in batch]),
        "attn_masks": torch.stack([b["attn_masks"] for b in batch]),
        "turn_mask":  torch.stack([b["turn_mask"]  for b in batch]),
        "n_turns":    torch.stack([b["n_turns"]    for b in batch]),
        "labels":     torch.stack([b["label"]      for b in batch]),
    }


def make_balanced_sampler(dataset: ScamMulticlassDataset) -> WeightedRandomSampler:
    """Balanced sampler: mỗi scenario được sample đều nhau."""
    labels = [int(dataset[i]["label"].item()) for i in range(len(dataset))]
    class_count = Counter(labels)
    weights = [1.0 / class_count[l] for l in labels]
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

## 5. Model Components

In [ ]:
class CrossTurnAttention(nn.Module):
    """Causal attention: turn t attends to turns 0..t-1."""

    def __init__(self, d_model: int, n_heads: int, dropout: float):
        super().__init__()
        self.mha = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads,
            dropout=dropout, batch_first=True,
        )
        self.norm = nn.LayerNorm(d_model)
        self.proj = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, h_seq: torch.Tensor, turn_mask: torch.Tensor) -> torch.Tensor:
        B, T, D = h_seq.shape
        out = torch.zeros_like(h_seq)
        out[:, 0, :] = h_seq[:, 0, :]
        for t in range(1, T):
            query  = h_seq[:, t:t+1, :]
            keys   = h_seq[:, :t, :]
            key_pm = ~turn_mask[:, :t]
            ctx, _ = self.mha(query, keys, keys, key_padding_mask=key_pm)
            fused  = self.proj(torch.cat([h_seq[:, t, :], ctx.squeeze(1)], dim=-1))
            out[:, t, :] = self.norm(fused + h_seq[:, t, :])
        return out

## 6. Loss Functions

In [ ]:
def hybrid_detection_loss(
    turn_probs: torch.Tensor,
    labels: torch.Tensor,
    turn_mask: torch.Tensor,
    w_max: float = 3.0,
    w_floor: float = 0.1,
    patience_weight: float = 0.5,
    class_weight_harmless: float = 8.0,
) -> torch.Tensor:
    """
    Binary streaming detection loss:
      First half  (t < N/2): soft-label BCE x w_floor  — phase evidence gathering
      Second half (t >= N/2): hard BCE x w_second       — phase commit (ramp 1->w_max)
      Patience: penalise p_t > p_{t+1} for scam dialogues
    """
    B, T = turn_probs.shape
    p = turn_probs.clamp(1e-6, 1 - 1e-6)

    n      = turn_mask.sum(dim=1).clamp(min=1).float()
    t_idx  = torch.arange(T, device=p.device).float()
    t_norm = t_idx.unsqueeze(0) / n.unsqueeze(1)   # [B, T]

    is_scam  = (labels == 1).unsqueeze(1).expand_as(p)
    is_first = (t_norm < 0.5)

    y_soft    = (2.0 * t_norm).clamp(max=1.0) * (labels == 1).float().unsqueeze(1)
    bce_first = -(y_soft * torch.log(p) + (1 - y_soft) * torch.log(1 - p))

    loss_second = torch.where(is_scam, -torch.log(p), -torch.log(1 - p))
    w_second    = 1.0 + (w_max - 1.0) * (2.0 * t_norm - 1.0).clamp(min=0.0, max=1.0)

    loss_t = torch.where(
        is_first, bce_first * w_floor, loss_second * w_second,
    ) * turn_mask.float()

    loss_per_sample = loss_t.sum(dim=1) / n
    cw = torch.where(
        labels == 0,
        torch.full_like(loss_per_sample, class_weight_harmless),
        torch.ones_like(loss_per_sample),
    )
    main_loss = (loss_per_sample * cw).mean()

    scam_b     = (labels == 1).float()
    pair_mask  = (turn_mask[:, :-1] & turn_mask[:, 1:]).float()
    n_pairs    = pair_mask.sum(dim=1).clamp(min=1)
    violations = F.relu(p[:, :-1] - p[:, 1:])
    patience_loss = ((violations * pair_mask).sum(dim=1) / n_pairs * scam_b).mean()

    return main_loss + patience_weight * patience_loss


def supervised_contrastive_loss(
    embeddings: torch.Tensor,
    labels: torch.Tensor,
    temperature: float = 0.07,
) -> torch.Tensor:
    """
    Supervised Contrastive Loss (Khosla et al. 2020).
    embeddings: [B, D] — L2-normalized
    labels:     [B]   — class indices
    Pulls embeddings of same class together, pushes different classes apart.
    """
    B = embeddings.size(0)
    if B < 2:
        return torch.tensor(0.0, device=embeddings.device, requires_grad=True)

    sim = embeddings @ embeddings.T / temperature   # [B, B]

    diag_mask = ~torch.eye(B, dtype=torch.bool, device=embeddings.device)
    pos_mask  = (labels.unsqueeze(0) == labels.unsqueeze(1)) & diag_mask

    if pos_mask.sum() == 0:
        return torch.tensor(0.0, device=embeddings.device, requires_grad=True)

    sim_max  = sim.max(dim=1, keepdim=True)[0].detach()
    exp_sim  = torch.exp(sim - sim_max) * diag_mask.float()
    log_prob = (sim - sim_max) - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)

    n_pos           = pos_mask.float().sum(dim=1).clamp(min=1)
    loss_per_anchor = -(pos_mask.float() * log_prob).sum(dim=1) / n_pos

    has_pos = (pos_mask.sum(dim=1) > 0).float()
    return (loss_per_anchor * has_pos).sum() / has_pos.sum().clamp(min=1)


def multiclass_contrastive_loss(
    embeddings: torch.Tensor,
    logits: torch.Tensor,
    labels: torch.Tensor,
    temperature: float = 0.07,
    alpha: float = 0.5,
) -> torch.Tensor:
    """
    alpha * SupConLoss(embeddings) + (1-alpha) * CrossEntropy(logits)
    """
    con_loss = supervised_contrastive_loss(embeddings, labels, temperature)
    ce_loss  = F.cross_entropy(logits, labels)
    return alpha * con_loss + (1.0 - alpha) * ce_loss

## 7. Models

In [ ]:
class BinaryM1Classifier(nn.Module):
    """HaLong per-turn encoder + causal CrossTurnAttention + binary sigmoid head."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.encoder = AutoModel.from_pretrained(cfg.model_name)
        self._frozen = True
        for p in self.encoder.parameters():
            p.requires_grad = False

        embed_dim = self.encoder.config.hidden_size
        d = cfg.hidden_dim
        self.proj = nn.Linear(embed_dim, d)
        self.attn = CrossTurnAttention(d, cfg.attn_heads, cfg.dropout)
        self.head = nn.Sequential(
            nn.Linear(d, d // 2), nn.GELU(),
            nn.Dropout(cfg.dropout), nn.Linear(d // 2, 1),
        )

    def unfreeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = True
        self._frozen = False
        if hasattr(self.encoder, "gradient_checkpointing_enable"):
            self.encoder.gradient_checkpointing_enable(
                gradient_checkpointing_kwargs={"use_reentrant": False}
            )

    def _encode_turns(self, input_ids, attn_masks):
        B, T, L = input_ids.shape
        cls_list = []
        for t in range(T):
            ids_t, mask_t = input_ids[:, t], attn_masks[:, t]
            if self._frozen:
                with torch.no_grad():
                    out = self.encoder(input_ids=ids_t, attention_mask=mask_t)
            else:
                out = self.encoder(input_ids=ids_t, attention_mask=mask_t)
            cls_list.append(out.last_hidden_state[:, 0])
            del out
        return torch.stack(cls_list, dim=1)   # [B, T, embed_dim]

    def forward(self, input_ids, attn_masks, turn_mask, labels=None):
        turn_mask = turn_mask.bool()
        n_turns   = turn_mask.sum(dim=1).long()
        B = input_ids.size(0)

        cls_seq = self._encode_turns(input_ids, attn_masks)
        h_seq   = F.gelu(self.proj(cls_seq))
        h_ctx   = self.attn(h_seq, turn_mask)
        logits  = self.head(h_ctx).squeeze(-1)   # [B, T]

        turn_probs     = torch.sigmoid(logits)
        dialogue_probs = torch.stack([turn_probs[b, n_turns[b]-1] for b in range(B)])

        loss = None
        if labels is not None:
            loss = hybrid_detection_loss(
                turn_probs, labels, turn_mask,
                w_max=self.cfg.w_max,
                w_floor=self.cfg.w_floor,
                patience_weight=self.cfg.patience_weight,
                class_weight_harmless=self.cfg.class_weight_harmless,
            )
        return {"loss": loss, "turn_probs": turn_probs, "dialogue_probs": dialogue_probs}

    def count_trainable_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


class ConversationMulticlassClassifier(nn.Module):
    """
    Multiclass per-conversation classifier cho scam scenario (A/B/C/D).
    Kiến trúc:
      HaLong encoder -> CrossTurnAttention -> last-turn pooling ->
        proj_head (cho contrastive loss) + cls_head (cho CE loss)
    Prediction: 1 nhãn duy nhất cho cả đoạn hội thoại.
    """

    def __init__(self, cfg: Config, num_classes: int):
        super().__init__()
        self.cfg = cfg
        self.num_classes = num_classes

        self.encoder = AutoModel.from_pretrained(cfg.model_name)
        self._frozen = True
        for p in self.encoder.parameters():
            p.requires_grad = False

        embed_dim = self.encoder.config.hidden_size
        d = cfg.hidden_dim
        self.proj = nn.Linear(embed_dim, d)
        self.attn = CrossTurnAttention(d, cfg.attn_heads, cfg.dropout)

        # Projection head: cho supervised contrastive loss
        self.proj_head = nn.Sequential(
            nn.Linear(d, d), nn.GELU(), nn.Linear(d, d // 2),
        )

        # Classification head: cho cross-entropy
        self.cls_head = nn.Sequential(
            nn.Linear(d, d // 2), nn.GELU(),
            nn.Dropout(cfg.dropout), nn.Linear(d // 2, num_classes),
        )

    def unfreeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = True
        self._frozen = False
        if hasattr(self.encoder, "gradient_checkpointing_enable"):
            self.encoder.gradient_checkpointing_enable(
                gradient_checkpointing_kwargs={"use_reentrant": False}
            )

    def _encode_turns(self, input_ids, attn_masks):
        B, T, L = input_ids.shape
        cls_list = []
        for t in range(T):
            ids_t, mask_t = input_ids[:, t], attn_masks[:, t]
            if self._frozen:
                with torch.no_grad():
                    out = self.encoder(input_ids=ids_t, attention_mask=mask_t)
            else:
                out = self.encoder(input_ids=ids_t, attention_mask=mask_t)
            cls_list.append(out.last_hidden_state[:, 0])
            del out
        return torch.stack(cls_list, dim=1)

    def forward(self, input_ids, attn_masks, turn_mask, labels=None):
        turn_mask = turn_mask.bool()
        n_turns   = turn_mask.sum(dim=1).long()
        B = input_ids.size(0)

        cls_seq  = self._encode_turns(input_ids, attn_masks)
        h_seq    = F.gelu(self.proj(cls_seq))
        h_ctx    = self.attn(h_seq, turn_mask)

        # Pool: lấy hidden state của turn cuối cùng hợp lệ
        conv_emb = torch.stack([h_ctx[b, n_turns[b]-1] for b in range(B)])  # [B, d]

        # Contrastive projection (L2-normalized)
        proj_emb = F.normalize(self.proj_head(conv_emb), dim=-1)  # [B, d//2]

        # Classification logits
        logits = self.cls_head(conv_emb)   # [B, num_classes]
        preds  = logits.argmax(dim=-1)     # [B]

        loss = None
        if labels is not None:
            loss = multiclass_contrastive_loss(
                proj_emb, logits, labels,
                temperature=self.cfg.contrastive_temp,
                alpha=self.cfg.contrastive_alpha,
            )
        return {"loss": loss, "logits": logits, "preds": preds, "embeddings": proj_emb}

    def count_trainable_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

## 8. Metrics

In [ ]:
def _first_alert_turn(turn_probs, threshold):
    for i, p in enumerate(turn_probs):
        if float(p) >= threshold:
            return i
    return None


def compute_streaming_metrics(all_labels, all_d_probs, all_t_probs, threshold=0.5):
    d_labels = np.array(all_labels)
    d_probs  = np.array(all_d_probs)
    d_preds  = (d_probs >= threshold).astype(int)

    try:
        auroc = float(roc_auc_score(d_labels, d_probs))
    except ValueError:
        auroc = float("nan")

    delays = []
    num_scam, num_det, num_harm, num_fa = 0, 0, 0, 0
    alert_turns, n_turns_list = [], []

    for lbl, tp in zip(all_labels, all_t_probs):
        first = _first_alert_turn(tp, threshold)
        alert_turns.append(first)
        n_turns_list.append(len(tp))
        if lbl == 1:
            num_scam += 1
            if first is not None:
                num_det += 1
                delays.append(first)
        else:
            num_harm += 1
            if first is not None:
                num_fa += 1

    tp_turns, tp_fracs = [], []
    for at, n, lbl in zip(alert_turns, n_turns_list, all_labels):
        if lbl == 1 and at is not None:
            tp_turns.append(at)
            tp_fracs.append(at / max(n, 1))

    early = {}
    if tp_turns:
        early = {
            "median_alert_turn": float(np.median(tp_turns)),
            "mean_alert_turn":   float(np.mean(tp_turns)),
            "median_lead_frac":  float(np.median(tp_fracs)),
            "alert_at_half":     float(np.mean([f <= 0.5 for f in tp_fracs])),
        }

    return {
        "dialogue_accuracy": float(accuracy_score(d_labels, d_preds)),
        "dialogue_f1":       float(f1_score(d_labels, d_preds, zero_division=0)),
        "auroc":             auroc,
        "detection_rate":      num_det / max(num_scam, 1),
        "avg_detection_delay": float(np.mean(delays)) if delays else float("nan"),
        "false_alarm_rate":    num_fa / max(num_harm, 1),
        "num_scam": num_scam, "num_harmless": num_harm,
        "num_detected": num_det, "num_false_alarms": num_fa,
        **early,
    }

## 9. Evaluation Functions

In [ ]:
@torch.no_grad()
def evaluate_binary(model, loader, device, threshold):
    model.eval()
    total_loss, n_dlg = 0.0, 0
    all_labels, all_d_probs, all_t_probs = [], [], []

    for batch in loader:
        ids    = batch["input_ids"].to(device)
        masks  = batch["attn_masks"].to(device)
        tmask  = batch["turn_mask"].to(device)
        labels = batch["labels"].to(device)
        n_turns = batch["n_turns"]

        out = model(ids, masks, tmask, labels=labels)
        B = labels.size(0)
        if out["loss"] is not None:
            total_loss += out["loss"].item() * B
        n_dlg += B

        for b in range(B):
            n = int(n_turns[b].item())
            all_labels.append(int(labels[b].item()))
            all_d_probs.append(float(out["dialogue_probs"][b].item()))
            all_t_probs.append(out["turn_probs"][b, :n].cpu().numpy())

        del out
        if device.type == "cuda":
            torch.cuda.empty_cache()

    m = compute_streaming_metrics(all_labels, all_d_probs, all_t_probs, threshold)
    m["loss"] = total_loss / max(n_dlg, 1)
    return m


@torch.no_grad()
def evaluate_multiclass(model, loader, device, scenario_map):
    """
    Evaluate multiclass model trên scam-only test set.
    Trả về accuracy per scenario + macro F1.
    """
    model.eval()
    total_loss, n_dlg = 0.0, 0
    all_preds, all_labels = [], []

    for batch in loader:
        ids    = batch["input_ids"].to(device)
        masks  = batch["attn_masks"].to(device)
        tmask  = batch["turn_mask"].to(device)
        labels = batch["labels"].to(device)

        out = model(ids, masks, tmask, labels=labels)
        B = labels.size(0)
        if out["loss"] is not None:
            total_loss += out["loss"].item() * B
        n_dlg += B
        all_preds.extend(out["preds"].cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

        del out
        if device.type == "cuda":
            torch.cuda.empty_cache()

    inv_map = {v: k for k, v in scenario_map.items()}
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    cnt_total   = Counter(all_labels)
    cnt_correct = Counter(p for p, l in zip(all_preds, all_labels) if p == l)
    per_class = {
        inv_map.get(c, str(c)): {
            "accuracy": cnt_correct[c] / cnt_total[c],
            "count":    cnt_total[c],
        }
        for c in sorted(cnt_total)
    }
    return {"loss": total_loss / max(n_dlg, 1), "accuracy": acc, "macro_f1": f1, "per_class": per_class}

## 10. Training Loop

In [ ]:
def run_binary_epoch(model, loader, optimizer, scheduler, device, cfg):
    model.train()
    total_loss, n_dlg = 0.0, 0
    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        ids    = batch["input_ids"].to(device)
        masks  = batch["attn_masks"].to(device)
        tmask  = batch["turn_mask"].to(device)
        labels = batch["labels"].to(device)

        out  = model(ids, masks, tmask, labels=labels)
        loss = out["loss"] / cfg.bin_grad_accum
        loss.backward()

        total_loss += out["loss"].item() * ids.size(0)
        n_dlg      += ids.size(0)

        if (step + 1) % cfg.bin_grad_accum == 0 or (step + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.bin_grad_clip)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        del out, loss
        if step % 20 == 0 and device.type == "cuda":
            torch.cuda.empty_cache()

        if (step + 1) % 10 == 0 or (step + 1) == len(loader):
            print(f"  [{step+1}/{len(loader)}] loss={total_loss/max(n_dlg,1):.4f} "
                  f"lr={scheduler.get_last_lr()[0]:.2e}")

    return total_loss / max(n_dlg, 1)


def run_mc_epoch(model, loader, optimizer, scheduler, device, cfg):
    model.train()
    total_loss, n_dlg = 0.0, 0
    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        ids    = batch["input_ids"].to(device)
        masks  = batch["attn_masks"].to(device)
        tmask  = batch["turn_mask"].to(device)
        labels = batch["labels"].to(device)

        out  = model(ids, masks, tmask, labels=labels)
        loss = out["loss"] / cfg.mc_grad_accum
        loss.backward()

        total_loss += out["loss"].item() * ids.size(0)
        n_dlg      += ids.size(0)

        if (step + 1) % cfg.mc_grad_accum == 0 or (step + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.mc_grad_clip)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        del out, loss
        if step % 20 == 0 and device.type == "cuda":
            torch.cuda.empty_cache()

    avg = total_loss / max(n_dlg, 1)
    print(f"  train_loss={avg:.4f} lr={scheduler.get_last_lr()[0]:.2e}")
    return avg

## 11. Report Helpers

In [ ]:
def print_binary_report(m, title="BINARY MODEL"):
    sep = "=" * 60
    print(f"\n{sep}\n{title}\n{sep}")
    print(f"  Accuracy:        {m['dialogue_accuracy']:.4f}")
    print(f"  F1:              {m['dialogue_f1']:.4f}")
    if not np.isnan(m.get("auroc", float("nan"))):
        print(f"  AUROC:           {m['auroc']:.4f}")
    print(f"  Detection rate:  {m['detection_rate']:.4f}  ({m['num_detected']}/{m['num_scam']})")
    if not np.isnan(m.get("avg_detection_delay", float("nan"))):
        print(f"  Avg delay:       {m['avg_detection_delay']:.2f} turns")
    print(f"  False alarm:     {m['false_alarm_rate']:.4f}  ({m['num_false_alarms']}/{m['num_harmless']})")
    if "median_alert_turn" in m:
        print(f"  Median alert:    turn {m['median_alert_turn']:.1f}  "
              f"(frac={m['median_lead_frac']:.3f}, in-1st-half={m['alert_at_half']:.3f})")
    print(sep)


def print_mc_report(m, title="MULTICLASS MODEL"):
    sep = "=" * 60
    print(f"\n{sep}\n{title}\n{sep}")
    print(f"  Accuracy:   {m['accuracy']:.4f}")
    print(f"  Macro F1:   {m['macro_f1']:.4f}")
    print("  Per-scenario:")
    for name, v in sorted(m["per_class"].items()):
        print(f"    Scenario {name}: {v['accuracy']:.4f}  (n={v['count']})")
    print(sep)

## 12. Stage 1 — Train Binary Model

In [ ]:
def train_binary(cfg, train_dlg, test_dlg, tokenizer):
    set_seed(cfg.seed)
    print("\n" + "=" * 60)
    print("STAGE 1: BINARY TRAINING")
    print("=" * 60)

    # Augmentation
    if cfg.truncate_aug:
        n_before  = len(train_dlg)
        train_dlg = truncate_augment(train_dlg, cfg.aug_k, cfg.aug_min_turns)
        print(f"Augmented: {n_before} -> {len(train_dlg)} dialogues")

    train_split, val_split = stratified_val_split(
        train_dlg, val_ratio=cfg.val_ratio, seed=cfg.seed
    )
    scam_tr = sum(1 for d in train_split if d["label"] == "scam")
    harm_tr = sum(1 for d in train_split if d["label"] == "harmless")
    print(f"Train: {len(train_split)} (scam={scam_tr}, harmless={harm_tr}) "
          f"| Val: {len(val_split)} | Test: {len(test_dlg)}")

    train_ds = BinaryDialogueDataset(train_split, tokenizer, cfg.max_turn_len, cfg.max_turns)
    val_ds   = BinaryDialogueDataset(val_split,   tokenizer, cfg.max_turn_len, cfg.max_turns)
    test_ds  = BinaryDialogueDataset(test_dlg,    tokenizer, cfg.max_turn_len, cfg.max_turns)

    train_loader = DataLoader(train_ds, batch_size=cfg.bin_batch_size, shuffle=True,
                              collate_fn=collate_fn, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,  batch_size=cfg.bin_batch_size, shuffle=False,
                              collate_fn=collate_fn, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds, batch_size=cfg.bin_batch_size, shuffle=False,
                              collate_fn=collate_fn, num_workers=2, pin_memory=True)

    model = BinaryM1Classifier(cfg).to(DEVICE)
    print(f"Trainable params (frozen encoder): {model.count_trainable_params():,}")

    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg.bin_lr, weight_decay=cfg.bin_weight_decay,
    )
    steps_per_epoch = max(1, len(train_loader) // cfg.bin_grad_accum)
    frozen_steps    = steps_per_epoch * max(1, cfg.bin_unfreeze_epoch - 1)
    warmup_steps    = max(1, int(frozen_steps * cfg.bin_warmup_ratio))
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=frozen_steps
    )

    save_dir = os.path.join(cfg.output_dir, "binary_model")
    os.makedirs(save_dir, exist_ok=True)
    best_acc, best_epoch, no_improve = -1.0, 0, 0

    for epoch in range(1, cfg.bin_epochs + 1):
        t0 = time.time()

        if epoch == cfg.bin_unfreeze_epoch:
            model.unfreeze_encoder()
            remaining = cfg.bin_epochs - epoch + 1
            total_uf  = steps_per_epoch * remaining
            optimizer = AdamW(model.parameters(), lr=cfg.bin_lr * 0.1,
                              weight_decay=cfg.bin_weight_decay)
            scheduler = get_cosine_schedule_with_warmup(
                optimizer,
                num_warmup_steps=max(1, int(total_uf * cfg.bin_warmup_ratio)),
                num_training_steps=total_uf,
            )
            print(f"\nEpoch {epoch}: encoder unfrozen, lr={cfg.bin_lr*0.1:.2e} | "
                  f"trainable={model.count_trainable_params():,}")

        print(f"\n--- Binary Epoch {epoch}/{cfg.bin_epochs} ---")
        tr_loss = run_binary_epoch(model, train_loader, optimizer, scheduler, DEVICE, cfg)
        val_m   = evaluate_binary(model, val_loader, DEVICE, cfg.binary_threshold)

        print(f"  t={time.time()-t0:.1f}s | tr_loss={tr_loss:.4f} "
              f"| val_acc={val_m['dialogue_accuracy']:.4f} "
              f"| val_f1={val_m['dialogue_f1']:.4f} "
              f"| dr={val_m['detection_rate']:.3f} "
              f"| far={val_m['false_alarm_rate']:.3f}")

        if val_m["dialogue_accuracy"] > best_acc:
            best_acc, best_epoch, no_improve = val_m["dialogue_accuracy"], epoch, 0
            torch.save(model.state_dict(), os.path.join(save_dir, "model.pt"))
            print(f"  * Saved best (val_acc={best_acc:.4f})")
        else:
            no_improve += 1
            if no_improve >= cfg.bin_patience:
                print(f"\nEarly stop at epoch {epoch}")
                break

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    # Load best & test
    model.load_state_dict(torch.load(
        os.path.join(save_dir, "model.pt"), map_location=DEVICE, weights_only=True
    ))
    print(f"\nBest binary epoch: {best_epoch} | val_acc={best_acc:.4f}")
    print_binary_report(evaluate_binary(model, test_loader, DEVICE, cfg.binary_threshold),
                        title="BINARY MODEL — TEST SET")
    return model

## 13. Stage 2 — Train Multiclass Model (scam-only, per-conversation)

In [ ]:
def train_multiclass(cfg, train_dlg, test_dlg, tokenizer, scenario_map):
    """
    Train multiclass scenario classifier:
    - Chi train tren scam dialogues (khong co harmless)
    - Loss: alpha*SupervisedContrastiveLoss + (1-alpha)*CrossEntropy
    - Prediction: 1 label duy nhat cho ca doan hoi thoai
    """
    set_seed(cfg.seed)
    print("\n" + "=" * 60)
    print("STAGE 2: MULTICLASS TRAINING")
    print(f"Scenario map: {scenario_map}")
    print(f"Loss: {cfg.contrastive_alpha:.1f}*SupCon + {1-cfg.contrastive_alpha:.1f}*CE  "
          f"(temp={cfg.contrastive_temp})")
    print("=" * 60)

    # Filter scam only
    scam_train = [d for d in train_dlg if d["label"] == "scam" and d.get("scenario") in scenario_map]
    scam_test  = [d for d in test_dlg  if d["label"] == "scam" and d.get("scenario") in scenario_map]

    # Augment
    if cfg.truncate_aug:
        n_before   = len(scam_train)
        scam_train = truncate_augment(scam_train, cfg.aug_k, cfg.aug_min_turns)
        print(f"Scam aug: {n_before} -> {len(scam_train)}")

    scam_train, scam_val = stratified_val_split(
        scam_train, val_ratio=cfg.val_ratio, seed=cfg.seed,
        label_fn=lambda d: d.get("scenario", "?")
    )
    print(f"Scam  Train: {len(scam_train)}  dist={dict(Counter(d['scenario'] for d in scam_train))}")
    print(f"Scam  Val:   {len(scam_val)}   dist={dict(Counter(d['scenario'] for d in scam_val))}")
    print(f"Scam  Test:  {len(scam_test)}")

    train_ds = ScamMulticlassDataset(scam_train, tokenizer, cfg.max_turn_len, cfg.max_turns, scenario_map)
    val_ds   = ScamMulticlassDataset(scam_val,   tokenizer, cfg.max_turn_len, cfg.max_turns, scenario_map)
    test_ds  = ScamMulticlassDataset(scam_test,  tokenizer, cfg.max_turn_len, cfg.max_turns, scenario_map)

    sampler = make_balanced_sampler(train_ds)
    train_loader = DataLoader(train_ds, batch_size=cfg.mc_batch_size, sampler=sampler,
                              collate_fn=collate_fn, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,  batch_size=cfg.mc_batch_size, shuffle=False,
                              collate_fn=collate_fn, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds, batch_size=cfg.mc_batch_size, shuffle=False,
                              collate_fn=collate_fn, num_workers=2, pin_memory=True)

    num_classes = len(scenario_map)
    model = ConversationMulticlassClassifier(cfg, num_classes).to(DEVICE)
    print(f"\nNum classes: {num_classes} | Trainable (frozen): {model.count_trainable_params():,}")

    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg.mc_lr, weight_decay=cfg.mc_weight_decay,
    )
    steps_per_epoch = max(1, len(train_loader) // cfg.mc_grad_accum)
    frozen_steps    = steps_per_epoch * max(1, cfg.mc_unfreeze_epoch - 1)
    warmup_steps    = max(1, int(frozen_steps * cfg.mc_warmup_ratio))
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=frozen_steps
    )

    save_dir = os.path.join(cfg.output_dir, "mc_model")
    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir, "scenario_map.json"), "w") as f:
        json.dump(scenario_map, f)

    best_acc, best_epoch, no_improve = -1.0, 0, 0

    for epoch in range(1, cfg.mc_epochs + 1):
        t0 = time.time()

        if epoch == cfg.mc_unfreeze_epoch:
            model.unfreeze_encoder()
            remaining = cfg.mc_epochs - epoch + 1
            total_uf  = steps_per_epoch * remaining
            optimizer = AdamW(model.parameters(), lr=cfg.mc_lr * 0.1,
                              weight_decay=cfg.mc_weight_decay)
            scheduler = get_cosine_schedule_with_warmup(
                optimizer,
                num_warmup_steps=max(1, int(total_uf * cfg.mc_warmup_ratio)),
                num_training_steps=total_uf,
            )
            print(f"\nEpoch {epoch}: MC encoder unfrozen, lr={cfg.mc_lr*0.1:.2e}")

        print(f"\n--- MC Epoch {epoch}/{cfg.mc_epochs} ---")
        tr_loss = run_mc_epoch(model, train_loader, optimizer, scheduler, DEVICE, cfg)
        val_m   = evaluate_multiclass(model, val_loader, DEVICE, scenario_map)

        print(f"  t={time.time()-t0:.1f}s | tr={tr_loss:.4f} "
              f"| val_acc={val_m['accuracy']:.4f} | macro_f1={val_m['macro_f1']:.4f}")
        for name, v in val_m["per_class"].items():
            print(f"    scenario {name}: acc={v['accuracy']:.3f} (n={v['count']})")

        if val_m["accuracy"] > best_acc:
            best_acc, best_epoch, no_improve = val_m["accuracy"], epoch, 0
            torch.save(model.state_dict(), os.path.join(save_dir, "model.pt"))
            print(f"  * Saved best (val_acc={best_acc:.4f})")
        else:
            no_improve += 1
            if no_improve >= cfg.mc_patience:
                print(f"\nEarly stop at epoch {epoch}")
                break

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    model.load_state_dict(torch.load(
        os.path.join(save_dir, "model.pt"), map_location=DEVICE, weights_only=True
    ))
    print(f"\nBest MC epoch: {best_epoch} | val_acc={best_acc:.4f}")
    print_mc_report(evaluate_multiclass(model, test_loader, DEVICE, scenario_map),
                    title="MULTICLASS MODEL (GT scam) — TEST SET")
    return model

## 14. Two-Stage Inference

Chỉ chạy multiclass khi binary dự đoán là **scam**.

In [ ]:
@torch.no_grad()
def two_stage_evaluate(binary_model, mc_model, test_dlg, tokenizer, cfg, scenario_map):
    """
    Full two-stage evaluation on test set:
    - Stage 1: binary model -> scam/harmless per dialogue
    - Stage 2: if binary=scam -> multiclass model -> scenario A/B/C/D
    In ra:
      - Binary metrics (tren toan bo test)
      - Multiclass metrics (chi tren dialogue binary=scam)
      - Phan phoi du doan scenario
    """
    binary_model.eval()
    mc_model.eval()
    inv_scenario = {v: k for k, v in scenario_map.items()}

    bin_gt, bin_pred, bin_prob = [], [], []
    # For MC: only dialogues where binary=scam
    mc_gt_scenario, mc_pred_scenario = [], []
    scenario_dist = Counter()

    # Process in batches
    full_ds     = BinaryDialogueDataset(test_dlg, tokenizer, cfg.max_turn_len, cfg.max_turns)
    full_loader = DataLoader(full_ds, batch_size=cfg.bin_batch_size, shuffle=False,
                             collate_fn=collate_fn, num_workers=2, pin_memory=True)

    dlg_idx = 0
    for batch in full_loader:
        ids    = batch["input_ids"].to(DEVICE)
        masks  = batch["attn_masks"].to(DEVICE)
        tmask  = batch["turn_mask"].to(DEVICE)
        labels = batch["labels"]  # binary GT
        B = ids.size(0)

        # Stage 1
        out_bin   = binary_model(ids, masks, tmask)
        b_probs   = out_bin["dialogue_probs"].cpu()
        b_preds   = (b_probs >= cfg.binary_threshold).long()

        bin_gt.extend(labels.tolist())
        bin_pred.extend(b_preds.tolist())
        bin_prob.extend(b_probs.tolist())

        # Stage 2: run MC on samples where binary=scam
        scam_idx_local = b_preds.nonzero(as_tuple=True)[0].tolist()
        if scam_idx_local:
            si = torch.tensor(scam_idx_local)
            out_mc = mc_model(
                ids[si], masks[si], tmask[si]
            )
            mc_preds = out_mc["preds"].cpu().tolist()
            for local_i, global_i in enumerate(scam_idx_local):
                sc_pred = inv_scenario.get(mc_preds[local_i], "?")
                scenario_dist[sc_pred] += 1
                # Record GT scenario if this is a real scam dialogue
                dlg = test_dlg[dlg_idx + global_i]
                gt_sc = dlg.get("scenario") if dlg["label"] == "scam" else None
                if gt_sc in scenario_map:
                    mc_gt_scenario.append(scenario_map[gt_sc])
                    mc_pred_scenario.append(mc_preds[local_i])

        del out_bin
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        dlg_idx += B

    # ── Report ─────────────────────────────────────────────────────────
    sep = "=" * 60
    print(f"\n{sep}\nTWO-STAGE PIPELINE — FULL TEST EVALUATION\n{sep}")

    # Binary
    bin_arr_gt   = np.array(bin_gt)
    bin_arr_pred = np.array(bin_pred)
    bin_arr_prob = np.array(bin_prob)
    try:
        auroc = roc_auc_score(bin_arr_gt, bin_arr_prob)
    except ValueError:
        auroc = float("nan")

    n_scam = (bin_arr_gt == 1).sum()
    n_harm = (bin_arr_gt == 0).sum()
    tp = ((bin_arr_pred == 1) & (bin_arr_gt == 1)).sum()
    fp = ((bin_arr_pred == 1) & (bin_arr_gt == 0)).sum()
    print("\nStage 1 — Binary:")
    print(f"  Accuracy: {accuracy_score(bin_arr_gt, bin_arr_pred):.4f}")
    print(f"  F1:       {f1_score(bin_arr_gt, bin_arr_pred, zero_division=0):.4f}")
    if not np.isnan(auroc):
        print(f"  AUROC:    {auroc:.4f}")
    print(f"  TPR:      {tp/max(n_scam,1):.4f}  ({tp}/{n_scam} scam detected)")
    print(f"  FPR:      {fp/max(n_harm,1):.4f}  ({fp}/{n_harm} harmless false-alarmed)")

    # Multiclass
    print(f"\nStage 2 — Multiclass (ran on {sum(bin_pred)} binary-scam dialogues):")
    print(f"  Scenario prediction distribution: {dict(sorted(scenario_dist.items()))}")
    if mc_pred_scenario:
        inv_sc = {v: k for k, v in scenario_map.items()}
        mc_acc = accuracy_score(mc_gt_scenario, mc_pred_scenario)
        mc_f1  = f1_score(mc_gt_scenario, mc_pred_scenario, average="macro", zero_division=0)
        print(f"  (On GT-scam-correctly-detected subset: n={len(mc_gt_scenario)})")
        print(f"  Accuracy:  {mc_acc:.4f}")
        print(f"  Macro F1:  {mc_f1:.4f}")
        cnt_tot = Counter(mc_gt_scenario)
        cnt_cor = Counter(p for p, l in zip(mc_pred_scenario, mc_gt_scenario) if p == l)
        for sc_idx in sorted(cnt_tot):
            sc_name = inv_sc.get(sc_idx, str(sc_idx))
            print(f"    Scenario {sc_name}: {cnt_cor[sc_idx]/cnt_tot[sc_idx]:.3f} ({cnt_tot[sc_idx]})")
    print(sep)

## 15. Main — Chạy toàn bộ pipeline

In [ ]:
# ────────────────────────────────────────────────────────────
# MAIN
# ────────────────────────────────────────────────────────────
set_seed(CFG.seed)

# 1. Load data
print("Loading data...")
train_dlg = []
for f in TRAIN_FILES:
    d = load_json(f)
    train_dlg.extend(d)
    n_sc = sum(1 for x in d if x["label"] == "scam")
    n_hm = sum(1 for x in d if x["label"] == "harmless")
    print(f"  {os.path.basename(f)}: {len(d)} dialogues (scam={n_sc}, harmless={n_hm})")

test_dlg = load_json(TEST_FILE)
print(f"  test: {len(test_dlg)} dialogues")

# 2. Scenario map (built from train, before augmentation)
scenario_map = build_scenario_map(train_dlg)
print(f"\nScenario map: {scenario_map}")

# 3. Tokenizer (shared by both models)
print(f"\nLoading tokenizer: {CFG.model_name}")
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)

# 4. Train binary model
binary_model = train_binary(CFG, train_dlg.copy(), test_dlg, tokenizer)

# 5. Train multiclass model
mc_model = train_multiclass(CFG, train_dlg.copy(), test_dlg, tokenizer, scenario_map)

# 6. Two-stage evaluation on full test set
two_stage_evaluate(binary_model, mc_model, test_dlg, tokenizer, CFG, scenario_map)

print(f"\nDone. Models saved to: {CFG.output_dir}")